In [7]:
import pandas as pd 
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('spotify_songs.csv')

df.shape
df.head(2)

features = ['danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness',
            'valence', 'tempo', 'duration_ms', 'key', 'mode']

X = df[features]
y = df['track_popularity']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

print(f"\nX_train shape: {X_train.shape}")
print(f"X_test  shape: {X_test.shape}")



X_train shape: (26266, 12)
X_test  shape: (6567, 12)


In [12]:
from sklearn.linear_model import Lasso
from sklearn.metrics import r2_score , mean_squared_error

lasso = Lasso(alpha = 1.0)
lasso.fit(X_train , y_train)

for feat , coef in zip(features , lasso.coef_):
    print(f"  {feat}: {'Removed (0)' if coef == 0 else f'{coef:.4f}'}")

y_pred = lasso.predict(X_test) 
print(f"\nR²  : {r2_score(y_test, y_pred):.4f}")
print(f"MSE : {mean_squared_error(y_test, y_pred):,.0f}") 

  danceability: 0.0961
  energy: -2.7466
  loudness: 1.7177
  speechiness: Removed (0)
  acousticness: 0.1446
  instrumentalness: -2.2530
  liveness: -0.0041
  valence: Removed (0)
  tempo: Removed (0)
  duration_ms: -2.1944
  key: Removed (0)
  mode: Removed (0)

R²  : 0.0562
MSE : 586


In [16]:
from sklearn.linear_model import Ridge 

ridge = Ridge(alpha=1)
ridge.fit(X_train , y_train)

for feat, lc, rc in zip(features, lasso.coef_, ridge.coef_):
    print(f"{feat}: Lasso={lc:.4f}, Ridge={rc:.4f}")

print(f"\nRidge Test R2: {ridge.score(X_test, y_test):.4f}")

print(f"\nLasso Test R2: {lasso.score(X_test, y_test):.4f}")

danceability: Lasso=0.0961, Ridge=0.7544
energy: Lasso=-2.7466, Ridge=-5.3052
loudness: Lasso=1.7177, Ridge=4.5286
speechiness: Lasso=-0.0000, Ridge=-0.7542
acousticness: Lasso=0.1446, Ridge=0.7818
instrumentalness: Lasso=-2.2530, Ridge=-2.6959
liveness: Lasso=-0.0041, Ridge=-0.6377
valence: Lasso=0.0000, Ridge=0.5975
tempo: Lasso=0.0000, Ridge=0.5609
duration_ms: Lasso=-2.1944, Ridge=-2.7377
key: Lasso=-0.0000, Ridge=0.0642
mode: Lasso=0.0000, Ridge=0.4608

Ridge Test R2: 0.0707

Lasso Test R2: 0.0562


In [23]:
import pandas as pd 
from sklearn.linear_model import LinearRegression, Ridge , Lasso , ElasticNet 

models = {
    'Linear'          : LinearRegression(),
    'Ridge  (L2)'     : Ridge(alpha=10),
    'Lasso  (L1)'     : Lasso(alpha=10),
    'ElasticNet(0.5)' : ElasticNet(alpha=10, l1_ratio=0.5),  # 50% L1 + 50% L2
    'ElasticNet(0.1)' : ElasticNet(alpha=10, l1_ratio=0.1),  # 10% L1 + 90% L2
    'ElasticNet(0.9)' : ElasticNet(alpha=10, l1_ratio=0.9),  # 90% L1 + 10% L2
}

for name , model in models.items():
    model.fit(X_train , y_train)
    coef = model.coef_
    r2 = r2_score(y_test , model.predict(X_test))

print(f"{name:<20} {coef[0]:>10.2f} {coef[1]:>10.2f} {coef[2]:>10.2f} {r2:>8.4f}")

ElasticNet(0.9)            0.00       0.00       0.00  -0.0002
